**Practical-2 :** Implement/Use the greedy 2-factor approximation algorithm for the Vertex Cover
problem for n = 10, m = 10, 20, 30 and 40.
a. Record the size of the computed vertex cover and the running time for each (n,
m) pair.
b. Compute the Approximation factor for each (n, m) pair using the results
obtained in Practical 1 above.

Taken help from GenAI model ChatGPT.




In [21]:
# Import libraries
import networkx as nx
import time
import itertools
import os
import csv
import matplotlib.pyplot as plt

In [22]:

# Parameters
n = 10  # number of vertices
edge_counts = list(range(10, 41, 10))  # m = 10, 20, 30, 40
input_dir = "/content"   # from Practical 1
output_dir = "output_results_practical2"  # new folder for comparison results

# Create directories if not exist
os.makedirs(output_dir, exist_ok=True)

In [23]:
# Check if a subset is a vertex cover
def is_vertex_cover(graph, cover):
    cover_set = set(cover)
    for u, v in graph.edges():
        if u not in cover_set and v not in cover_set:
            return False
    return True

# Brute force: Minimum Vertex Cover
def brute_force_vertex_cover(graph):
    nodes = list(graph.nodes())
    for r in range(1, len(nodes) + 1):
        for subset in itertools.combinations(nodes, r):
            if is_vertex_cover(graph, subset):
                return list(subset)
    return nodes  # worst case
# Greedy 2-Approximation Algorithm
def greedy_vertex_cover(graph):
    cover = set()
    edges = set(graph.edges())
    while edges:
        (u, v) = edges.pop()
        cover.add(u)
        cover.add(v)
        # remove edges incident on u or v
        edges = {e for e in edges if u not in e and v not in e}
    return list(cover)

def find_maximum_matching(graph):
    matching = nx.max_weight_matching(graph, maxcardinality=True, weight=None)
    matching_edges = list(matching)
    matching_vertices = set(u for edge in matching_edges for u in edge)
    return matching_edges, matching_vertices

import pandas as pd

results = []  # store all results

edges_list = []
time_brute = []
time_greedy = []
approx_factors = []

for m in edge_counts:
    # Load the graph from the uploaded CSV file
    input_filename = f"{input_dir}/input_graph_n{n}_m{m} (1).csv"
    edges_df = pd.read_csv(input_filename)
    G = nx.from_pandas_edgelist(edges_df, source='node1', target='node2')


    # --- Brute Force (OPT) ---
    start = time.perf_counter()
    cover_opt = brute_force_vertex_cover(G)
    end = time.perf_counter()
    time_opt = end - start
    size_opt = len(cover_opt)

    # --- Greedy (2-Approx) ---
    start = time.perf_counter()
    cover_greedy = greedy_vertex_cover(G)
    end = time.perf_counter()
    time_g = end - start
    size_g = len(cover_greedy)

    # Approximation factor
    approx_factor = round(size_g / size_opt, 2)

    # Store results
    results.append([
        n, m,
        size_opt, cover_opt, round(time_opt, 6),
        size_g, cover_greedy, round(time_g, 6),
        approx_factor
    ])

    edges_list.append(m)
    time_brute.append(time_opt)
    time_greedy.append(time_g)
    approx_factors.append(approx_factor)

    print(f"Done (n={n}, m={m}): OPT={size_opt}, Greedy={size_g}, Factor={approx_factor}")

csv_filename = f"{output_dir}/vertex_cover_comparison.csv"
with open(csv_filename, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([
        "n", "m",
        "opt_cover_size", "opt_vertices", "opt_time_sec",
        "greedy_cover_size", "greedy_vertices", "greedy_time_sec",
        "approx_factor"
    ])
    writer.writerows(results)

print(f"\nConsolidated results saved to {csv_filename}")

Done (n=10, m=10): OPT=3, Greedy=4, Factor=1.33
Done (n=10, m=20): OPT=6, Greedy=8, Factor=1.33
Done (n=10, m=30): OPT=7, Greedy=8, Factor=1.14
Done (n=10, m=40): OPT=8, Greedy=10, Factor=1.25

Consolidated results saved to output_results_practical2/vertex_cover_comparison.csv


In [24]:
# 1. Execution time comparison
plt.figure(figsize=(8, 6))
plt.plot(edges_list, time_brute, marker="o", label="Brute Force")
plt.plot(edges_list, time_greedy, marker="s", label="Greedy 2-Approx")
plt.xlabel("Number of Edges (m)")
plt.ylabel("Execution Time (seconds)")
plt.title(f"Execution Time vs. Number of Edges (n={n})")
plt.legend()
plt.grid(True)
plt.savefig(f"{output_dir}/execution_time_comparison.png")
plt.close()

# 2. Approximation factor vs edges
plt.figure(figsize=(8, 6))
plt.plot(edges_list, approx_factors, marker="d", color="purple")
plt.xlabel("Number of Edges (m)")
plt.ylabel("Approximation Factor (Greedy/OPT)")
plt.title(f"Approximation Factor vs. Edges (n={n})")
plt.grid(True)
plt.savefig(f"{output_dir}/approximation_factor_vs_edges.png")
plt.close()

print(f"Plots saved in {output_dir}")

Plots saved in output_results_practical2
